# NEOFC - Prepare connectivity data

In [ ]:
from pathlib import Path 
import pandas as pd 
import numpy as np 
import xarray as xr 
import pickle
import gzip 
from scipy.stats import zscore
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from mapconn.matrix import _precision_to_partial_pearson, matrix_sac

# working directory
wd = Path.cwd() 
print(wd)

# general vars
from utils import get_dist_mat
from utils import (PARCS_ALL, PARCS_CX, PARC_DEFAULT,
                   MEG_MEASURES_ALL, MEG_FQBANDS)

# subcortical parcels
sc_parcels = [
    'THALAMUS_LEFT', 
    'CAUDATE_LEFT', 
    'PUTAMEN_LEFT',
    'PALLIDUM_LEFT', 
    'HIPPOCAMPUS_LEFT',
    'AMYGDALA_LEFT',
    'ACCUMBENS_LEFT', 
    'DIENCEPHALON_VENTRAL_LEFT', 
    'THALAMUS_RIGHT', 
    'CAUDATE_RIGHT', 
    'PUTAMEN_RIGHT',
    'PALLIDUM_RIGHT', 
    'HIPPOCAMPUS_RIGHT', 
    'AMYGDALA_RIGHT',
    'ACCUMBENS_RIGHT', 
    'DIENCEPHALON_VENTRAL_RIGHT'
]

/Users/llotter/projects/mapfc


## Distance matrices

In [3]:
distmat = get_dist_mat(PARCS_ALL)
display(distmat[PARC_DEFAULT].head())

,hemi-L_div-Vis_lab-1,hemi-L_div-Vis_lab-2,hemi-L_div-Vis_lab-3,hemi-L_div-Vis_lab-4,hemi-L_div-Vis_lab-5,hemi-L_div-Vis_lab-6,hemi-L_div-Vis_lab-7,hemi-L_div-Vis_lab-8,hemi-L_div-Vis_lab-9,hemi-L_div-Vis_lab-10,...,hemi-R_div-Default_lab-PFCm+1,hemi-R_div-Default_lab-PFCm+2,hemi-R_div-Default_lab-PFCm+3,hemi-R_div-Default_lab-PFCm+4,hemi-R_div-Default_lab-PFCm+5,hemi-R_div-Default_lab-PFCm+6,hemi-R_div-Default_lab-PFCm+7,hemi-R_div-Default_lab-PCC+1,hemi-R_div-Default_lab-PCC+2,hemi-R_div-Default_lab-PCC+3
label,,,,,,,,,,,,,,,,,,,,,
hemi-L_div-Vis_lab-1,0.000000,27.188728,29.195051,22.916956,44.016490,16.922497,45.137010,36.328793,47.806670,29.319662,...,94.066376,101.51171,90.776180,119.48091,119.819885,111.540764,110.41165,45.27642,52.123220,62.846170
hemi-L_div-Vis_lab-2,27.188728,0.000000,25.525430,25.109753,23.910486,37.377876,29.891562,35.300842,31.551050,30.642378,...,117.398260,125.29286,114.422905,143.30680,142.461320,132.846270,131.17854,54.27349,63.042580,70.017420
hemi-L_div-Vis_lab-3,29.195051,25.525430,0.000000,38.107502,33.827980,41.035397,47.922745,21.789507,40.158447,39.943413,...,117.336610,124.54478,113.603110,141.44849,141.177080,134.385510,131.91997,65.36880,69.498410,75.683330
hemi-L_div-Vis_lab-4,22.916956,25.109753,38.107502,0.000000,36.694595,23.850756,28.197859,42.153805,36.663155,18.997562,...,104.588680,110.95808,99.160690,128.59416,126.027220,114.757324,113.19505,34.37658,44.655933,53.608673
hemi-L_div-Vis_lab-5,44.016490,23.910486,33.827980,36.694595,0.000000,53.941980,26.491129,40.889862,23.680370,38.927856,...,135.222080,142.80603,131.459800,160.50435,158.294920,147.864270,145.57732,64.25235,72.783775,76.509160


## General functions

In [ ]:
def window_timeseries(timeseries_xarray, subindex_key, tr, window_size_seconds=60, step_size_seconds=10):
    
    assert isinstance(timeseries_xarray, xr.DataArray), "timeseries_xarray must be an xarray.DataArray"
    assert "sub" in timeseries_xarray.coords, "timeseries_xarray must have a 'sub' dimension"
    assert "tp" in timeseries_xarray.coords, "timeseries_xarray must have a 'tp' dimension"
    assert subindex_key in timeseries_xarray.coords, f"timeseries_xarray must have a '{subindex_key}' dimension"
    
    # get window size in trs
    n_tp = timeseries_xarray[0].coords["tp"].shape[0]
    window_size = np.ceil(window_size_seconds / tr).astype(int)
    step_size = np.ceil(step_size_seconds / tr).astype(int)
    print(f"{window_size_seconds} s windows with {step_size_seconds} s steps @ tr = {tr}s: {window_size} elements per window and {step_size} element steps")
    
    # list for new array
    new_timeseries_arr = []
    
    # iterate windows
    coords_subindex = []
    coords_tp = np.arange(window_size+1)
    coords_parcel = timeseries_xarray.coords["parcel"].values
    i_start_seconds = 0
    for i_start in range(0, n_tp - window_size, step_size):
        xarr_step = timeseries_xarray.sel(tp=slice(i_start, i_start+window_size)).copy()
        coords_subindex.append([
            (f"{idx[0]}_win-{i_start_seconds}", ) + tuple(list(idx)[1:]) 
            for idx in xarr_step.coords[subindex_key].values
        ])
        new_timeseries_arr.append(xarr_step.values)
        i_start_seconds += step_size_seconds
    
    # concat
    new_timeseries_arr = xr.DataArray(
        data=np.concatenate(new_timeseries_arr, axis=0),
        coords={
            subindex_key: pd.MultiIndex.from_tuples(sum(coords_subindex, []), names=subindex_key.split("-")),
            "tp": coords_tp,
            "parcel": coords_parcel,
        }
    )
    return new_timeseries_arr


def calculate_sac(conn_xarray, distmat):
    sac = {}
    for measure in conn_xarray.coords[conn_xarray.dims[0]].values:
        for idx in conn_xarray.coords[conn_xarray.dims[1]].values:
            
            # individual matrix
            mat = conn_xarray.loc[measure, idx, :, :].values.copy()
            
            # get gc
            gc = np.nanmean(mat[np.triu_indices(mat.shape[0], k=1)])
            
            # get sac
            mat = np.tanh(mat)
            np.fill_diagonal(mat, 1)
            k = (measure,) + (idx if isinstance(idx, tuple) else (idx,))
            sac[k] = matrix_sac(mat, distmat=distmat.values, discretization="fd") + (gc,)
                
    # to frame
    sac = (
        pd.DataFrame(sac).T
        .rename(columns={0: "sa_lambda", 1: "sa_inf", 2: "gc"})
        .rename_axis(index=[conn_xarray.dims[0], *conn_xarray.dims[1].split("-")])
    )
    return sac
            
            
def connectivity_from_timeseries(timeseries_xarray, 
                                 connectivity_measures=["pearson", ("degreecentrality", 0.2), ("shrunkprec", 0.1)],
                                 n_jobs=-1,
                                 dtype=np.float16,
                                 hcp_average_by_dir=True,
                                 calculate_autocorrelation=False,
                                 distmat=None):
    
    """Function to compute connectivity from timeseries xarray"""
    
    # if timeseries_xarray is a list: concatenate in order across last dimension!
    if isinstance(timeseries_xarray, list):
        print("timeseries_xarray is a list, concatenating in across dimension 'parcel'")
        timeseries_xarray = xr.concat(timeseries_xarray, dim="parcel")
    
    # check arr
    if not isinstance(timeseries_xarray, xr.DataArray):
        raise ValueError("timeseries_xarray must be an xarray DataArray")
    if not len(timeseries_xarray.dims) == 3:
        raise ValueError("timeseries_xarray must have 3 dimensions")
    print("timeseries_xarray.shape:", timeseries_xarray.shape)
    xarray_dims = timeseries_xarray.dims
    xarray_index_coords = xarray_dims[0].split("-")
    xarray_coords = list(timeseries_xarray.coords)
    print("xarray_dims:", xarray_dims)
    print("xarray_coords:", xarray_coords)  
    if not all(k in xarray_coords for k in ["sub", "parcel", "tp"]):
        raise ValueError("timeseries_xarray must have 'sub','parcel' and 'tp' coordinates")
    print("xarray_index_coords:", xarray_index_coords)
    
    # check connectivity measures
    if isinstance(connectivity_measures, str):
        connectivity_measures = [connectivity_measures]
    
    # autocorr
    if calculate_autocorrelation and distmat is None:
        raise ValueError("distmat must be provided if calculate_autocorrelation is True")
    
    # we will iterate over subjects and, if provided, every other dimension that is not "sub" or "dir"
    # "dir" is direction and will be averaged for each subject per dimension
    # other dimensions are, e.g.: run or treatment
    index_to_iterate = timeseries_xarray.coords[xarray_dims[0]].to_pandas().index
    if hcp_average_by_dir:
        average_by_dir = True if "dir" in xarray_index_coords else False
    else:
        average_by_dir = False
    print("average_by_dir:", average_by_dir)
    if average_by_dir:
        index_to_iterate = index_to_iterate.droplevel("dir")
    index_to_iterate = index_to_iterate.unique()
    print("index_to_iterate:", index_to_iterate[:5])
        
    # list to store connectivity
    conn_list = []
    conn_list_names = []
    
    # iterate over connectivity measures
    for conn_measure in connectivity_measures:
        
        # get measure and parameter
        if isinstance(conn_measure, tuple):
            conn_measure, param = conn_measure
        else:
            param = None
        print("Connectivity measure:", conn_measure, "with parameter:", param)
        if conn_measure in ["shrunkprec", "pearsonthresh"] and param is None:
            raise ValueError("Parameter must be provided for shrinkage / binary measures")
        
        # calculate connectivity
        # parallel function
        def par_fun(timeseries_xarray, idx, print_checks=False):
            
            # get timeseries for one index (will be a tuple)
            if not average_by_dir:
                ts = timeseries_xarray.loc[idx, :, :].values,
                if print_checks:
                    print(f"{conn_measure} | {idx} | ts:", ts[0].shape)
            else:
                ts = (timeseries_xarray.loc[idx + ("RL",), :, :].values,
                      timeseries_xarray.loc[idx + ("LR",), :, :].values)
                if print_checks:
                    print(f"{conn_measure} | {idx} | ts (RL, LR):", ts[0].shape, ts[1].shape)
                    
            # calculate connectivity
            conn = []
            for ts_dir in ts:
                
                # pearson
                if conn_measure == "pearson":
                    mat = np.corrcoef(ts_dir.T)
                    # r to z
                    with np.errstate(divide='ignore'):
                        mat = np.arctanh(mat)
                        
                # pearsonthresh
                elif conn_measure == "pearsonthresh":
                    mat = np.corrcoef(ts_dir.T)
                    mat[mat < param] = np.nan
                    # r to z
                    with np.errstate(divide='ignore'):
                        mat = np.arctanh(mat)
                    
                else:
                    raise ValueError(f"Connectivity measure {conn_measure} not supported")
                
                if print_checks:
                        print(f"{conn_measure} | {idx} | mat:", mat.shape)
                        #print(mat)
                
                # save
                conn.append(mat)
                
            conn = np.mean(conn, axis=0)
            if print_checks:
                print(f"{conn_measure} | {idx} | conn:", conn.shape)

            return conn
            
            
        # parallelize
        conn_list_measure = Parallel(n_jobs=n_jobs)(
            delayed(par_fun)(timeseries_xarray, idx, print_checks=True if i == 0 else False) 
            for i, idx in enumerate(tqdm(index_to_iterate))
        ) 
        conn_list.append(conn_list_measure)
        conn_list_names.append(conn_measure)
        
    # into 4d array
    conn_xarray = xr.DataArray(
        data=conn_list,
        coords={
            "measure": conn_list_names,
            "-".join(index_to_iterate.names): index_to_iterate,
            "parcelx": timeseries_xarray.coords["parcel"].values,
            "parcely": timeseries_xarray.coords["parcel"].values
        }
    )
    conn_xarray = conn_xarray.astype(dtype)
    
    # spatial autocorrelation
    if calculate_autocorrelation:
        sac = calculate_sac(conn_xarray, distmat)
        return conn_xarray, sac

    return conn_xarray
          

# test_arr = pickle.load(gzip.open(wd / "data_source" / "timeseries" / "hcp_ya_mri" / f"seg-Schaefer100_timeseries.pkl.gz", "rb"))
# test_conn, sac = connectivity_from_timeseries(timeseries_xarray=test_arr, n_jobs=-1, calculate_autocorrelation=True, distmat=distmat["Schaefer100"])
# sac

## HCP-YA: MRI

### Subjects

In [8]:
# Preselected subjects
subs_hcp_ya = np.loadtxt(wd / "data_source" / "pheno" / "hcp_ya" / "hcp_ya_subjects.txt", dtype=int)

# QC data 
qc_ya_mri = pd.read_table(wd / "data_source" / "timeseries" / "hcp_ya_mri" / "linc_qc.tsv")
qc_ya_mri = qc_ya_mri.set_index(["sub", "run"], drop=False)
qc_ya_mri = qc_ya_mri.loc[qc_ya_mri.index.get_level_values("sub").isin(subs_hcp_ya)]
print(f"MRI data for {len(qc_ya_mri['sub'].unique())}/{len(subs_hcp_ya)} subjects")
#display(qc_ya_mri.head())
display(qc_ya_mri["mean_fd"].describe())

# get subs with mean_fd <= 0.3 for both directions in both runs
subs_ya_mri = [
    sub for sub in qc_ya_mri["sub"].unique() 
    if (qc_ya_mri.loc[(sub, slice(None)), "mean_fd"] <= 0.3).all()
]
print(f"{len(subs_ya_mri)}/{len(qc_ya_mri['sub'].unique())} subs with mean_fd <= 0.3 for both directions")

MRI data for 124/132 subjects


count    496.000000
mean       0.164915
std        0.063437
min        0.075132
25%        0.124779
50%        0.148499
75%        0.186211
max        0.638250
Name: mean_fd, dtype: float64

112/124 subs with mean_fd <= 0.3 for both directions


### Connectomes

In [5]:
for parc in PARCS_ALL:
    print(parc)
    
    # load timeseries arrays for cortical and subcortical parcellations
    if parc in PARCS_CX:
        timeseries_xarray = (
            pickle.load(gzip.open(wd / "data_source" / "timeseries" / "hcp_ya_mri" / f"seg-{parc}_timeseries.pkl.gz", "rb"))
            .loc[([f"sub-{sub}" for sub in subs_ya_mri], slice(None), slice(None)), :, :]
        )
        
    # for whole-brain parcellation, load both cortical and subcortical timeseries arrays
    else:
        timeseries_xarray = [
            (
                pickle.load(gzip.open(wd / "data_source" / "timeseries" / "hcp_ya_mri" / f"seg-{p}_timeseries.pkl.gz", "rb"))
                .loc[([f"sub-{sub}" for sub in subs_ya_mri], slice(None), slice(None)), :, sc_parcels if p == "HCP" else slice(None)]
            )
            for p in [parc.replace("Subcortical", ""), "HCP"]
        ]
        
    # calculate connectivity
    conn, sac = connectivity_from_timeseries(
        timeseries_xarray=timeseries_xarray, 
        connectivity_measures="pearson",
        distmat=distmat[parc],
        calculate_autocorrelation=True
    )
    print(conn.shape)
    print("")
    
    # save
    with gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ya_mri" / f"parc-{parc}_connectivity.pkl.gz", "wb") as f:
        pickle.dump(conn, f)
    sac.to_csv(wd / "data_deriv" / "connectomes" / "hcp_ya_mri" / f"parc-{parc}_sac.csv")
        
    # NO LR/RL AVERAGING
    # cases in which this will be calculated: 
    if "Schaefer200" in parc:
        print("No LR-RL direction averaging")

        # calculate connectivity
        conn, sac = connectivity_from_timeseries(
            timeseries_xarray=timeseries_xarray, 
            hcp_average_by_dir=False,
            connectivity_measures="pearson",
            distmat=distmat[parc], 
            calculate_autocorrelation=True
        )
        print(conn.shape)
        print("")

        # save
        with gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ya_mri" / f"parc-{parc}_connectivity_nodiravg.pkl.gz", "wb") as f:
            pickle.dump(conn, f)
        sac.to_csv(wd / "data_deriv" / "connectomes" / "hcp_ya_mri" / f"parc-{parc}_sac_nodiravg.csv")

Schaefer100
timeseries_xarray.shape: (448, 1200, 100)
xarray_dims: ('sub-run-dir', 'tp', 'parcel')
xarray_coords: ['sub-run-dir', 'sub', 'run', 'dir', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run', 'dir']
average_by_dir: True
index_to_iterate: MultiIndex([('sub-100408', '1'),
            ('sub-100408', '2'),
            ('sub-101309', '1'),
            ('sub-101309', '2'),
            ('sub-101915', '1')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/224 [00:00<?, ?it/s]

pearson | ('sub-100408', '1') | ts (RL, LR): (1200, 100) (1200, 100)
pearson | ('sub-100408', '1') | mat: (100, 100)
pearson | ('sub-100408', '1') | mat: (100, 100)
pearson | ('sub-100408', '1') | conn: (100, 100)
(1, 224, 100, 100)

Schaefer100Subcortical
timeseries_xarray is a list, concatenating in across dimension 'parcel'
timeseries_xarray.shape: (448, 1200, 116)
xarray_dims: ('sub-run-dir', 'tp', 'parcel')
xarray_coords: ['sub-run-dir', 'sub', 'run', 'dir', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run', 'dir']
average_by_dir: True
index_to_iterate: MultiIndex([('sub-100408', '1'),
            ('sub-100408', '2'),
            ('sub-101309', '1'),
            ('sub-101309', '2'),
            ('sub-101915', '1')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/224 [00:00<?, ?it/s]

pearson | ('sub-100408', '1') | ts (RL, LR): (1200, 116) (1200, 116)
pearson | ('sub-100408', '1') | mat: (116, 116)
pearson | ('sub-100408', '1') | mat: (116, 116)
pearson | ('sub-100408', '1') | conn: (116, 116)
(1, 224, 116, 116)

Schaefer200
timeseries_xarray.shape: (448, 1200, 200)
xarray_dims: ('sub-run-dir', 'tp', 'parcel')
xarray_coords: ['sub-run-dir', 'sub', 'run', 'dir', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run', 'dir']
average_by_dir: True
index_to_iterate: MultiIndex([('sub-100408', '1'),
            ('sub-100408', '2'),
            ('sub-101309', '1'),
            ('sub-101309', '2'),
            ('sub-101915', '1')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


...

pearson | ('sub-100408', '1', 'LR') | ts: (1200, 216)
pearson | ('sub-100408', '1', 'LR') | mat: (216, 216)
pearson | ('sub-100408', '1', 'LR') | conn: (216, 216)
(1, 448, 216, 216)

Schaefer400
timeseries_xarray.shape: (448, 1200, 400)
xarray_dims: ('sub-run-dir', 'tp', 'parcel')
xarray_coords: ['sub-run-dir', 'sub', 'run', 'dir', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run', 'dir']
average_by_dir: True
index_to_iterate: MultiIndex([('sub-100408', '1'),
            ('sub-100408', '2'),
            ('sub-101309', '1'),
            ('sub-101309', '2'),
            ('sub-101915', '1')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/224 [00:00<?, ?it/s]

pearson | ('sub-100408', '1') | ts (RL, LR): (1200, 400) (1200, 400)
pearson | ('sub-100408', '1') | mat: (400, 400)
pearson | ('sub-100408', '1') | mat: (400, 400)
pearson | ('sub-100408', '1') | conn: (400, 400)
(1, 224, 400, 400)

Schaefer400Subcortical
timeseries_xarray is a list, concatenating in across dimension 'parcel'
timeseries_xarray.shape: (448, 1200, 416)
xarray_dims: ('sub-run-dir', 'tp', 'parcel')
xarray_coords: ['sub-run-dir', 'sub', 'run', 'dir', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run', 'dir']
average_by_dir: True
index_to_iterate: MultiIndex([('sub-100408', '1'),
            ('sub-100408', '2'),
            ('sub-101309', '1'),
            ('sub-101309', '2'),
            ('sub-101915', '1')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/224 [00:00<?, ?it/s]

pearson | ('sub-100408', '1') | ts (RL, LR): (1200, 416) (1200, 416)
pearson | ('sub-100408', '1') | mat: (416, 416)
pearson | ('sub-100408', '1') | mat: (416, 416)
pearson | ('sub-100408', '1') | conn: (416, 416)
(1, 224, 416, 416)



## HCP-YA: MEG

In [9]:
subs_ya_meg = sorted([
    f.name 
    for f in (wd / "data_deriv" / "connectomes" / "hcp_ya_meg" / "parcelAECConnectivity_Schaefer200").glob("*")
    if f.is_dir()
])

print(f"{len(subs_ya_meg)} subjects: {subs_ya_meg[:5]} ...")

33 subjects: ['102816', '106521', '109123', '113922', '116524'] ...


### Sort connectomes

These were provided by Bratislav Misic and Golia Shafiei as matrices. We will just pack them into xarray

In [10]:
# iterate over parcs
for parc in PARCS_CX:
    print(parc)
    
    # get labels
    parc_labels = pd.read_csv(wd / "parcellation" / f"parc-{parc}.label.csv", index_col=0)["label"].to_list()
    
    # data
    data_ya_meg = {}
    
    # iterate over measures
    for measure in MEG_MEASURES_ALL:
        data_ya_meg[measure] = {}
        if measure == "aec":
            measure_dir = f"parcelAECConnectivity_{parc}"
        elif measure == "aecorth":
            measure_dir = f"parcelAECConnectivity_orth_{parc}"
        else:
            raise ValueError(f"Measure {measure} not supported")
        
        # iterate over fqbands
        for fq in MEG_FQBANDS:
            
            # iterate over subjects
            for sub in subs_ya_meg:        
                
                data = np.load(
                    wd / "data_deriv" / "connectomes" / "hcp_ya_meg" / measure_dir / sub / f"{sub}_aecConn_{fq}_{parc}.npy"
                )
                
                # Fisher z transform
                with np.errstate(divide='ignore'):
                    data = np.arctanh(data)
            
                # save
                data_ya_meg[measure][f"sub-{sub}", fq] = data
                
    # to xarray
    data_ya_meg = xr.DataArray(
        data=[[mat for mat in data_ya_meg[measure].values()] for measure in MEG_MEASURES_ALL],
        coords={
            "measure": MEG_MEASURES_ALL,
            "sub-fqband": pd.MultiIndex.from_tuples(data_ya_meg["aec"].keys(), names=["sub", "fqband"]),
            "parcelx": parc_labels,
            "parcely": parc_labels
        }
    )
    data_ya_meg = data_ya_meg.astype(np.float16)
    print(data_ya_meg.shape)
    print(data_ya_meg.coords)
    
    # save
    with gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ya_meg" / f"parc-{parc}_connectivity.pkl.gz", "wb") as f:
        pickle.dump(data_ya_meg, f)
        
    # spatial autocorrelation
    sac = calculate_sac(data_ya_meg, distmat=distmat[parc])
    sac.to_csv(wd / "data_deriv" / "connectomes" / "hcp_ya_meg" / f"parc-{parc}_sac.csv")

Schaefer100
(2, 198, 100, 100)
Coordinates:
  * measure     (measure) <U7 56B 'aec' 'aecorth'
  * sub-fqband  (sub-fqband) object 2kB MultiIndex
  * sub         (sub-fqband) object 2kB 'sub-102816' ... 'sub-891667'
  * fqband      (sub-fqband) object 2kB 'delta' 'delta' ... 'hgamma' 'hgamma'
  * parcelx     (parcelx) <U39 16kB 'hemi-L_div-Vis_lab-1' ... 'hemi-R_div-De...
  * parcely     (parcely) <U39 16kB 'hemi-L_div-Vis_lab-1' ... 'hemi-R_div-De...
Schaefer200
(2, 198, 200, 200)
Coordinates:
  * measure     (measure) <U7 56B 'aec' 'aecorth'
  * sub-fqband  (sub-fqband) object 2kB MultiIndex
  * sub         (sub-fqband) object 2kB 'sub-102816' ... 'sub-891667'
  * fqband      (sub-fqband) object 2kB 'delta' 'delta' ... 'hgamma' 'hgamma'
  * parcelx     (parcelx) <U39 31kB 'hemi-L_div-Vis_lab-1' ... 'hemi-R_div-De...
  * parcely     (parcely) <U39 31kB 'hemi-L_div-Vis_lab-1' ... 'hemi-R_div-De...
Schaefer400
(2, 198, 400, 400)
Coordinates:
  * measure     (measure) <U7 56B 'aec' 'aecor

## PHYSIO: YRSP dataset

"Yale Resting State fMRI/Pupillometry: Arousal Study"

### Select subjects / sessions

In [8]:
# QC data 
linc_qc = (
      pd.read_table(wd / "data_source" / "timeseries" / "yrsp" / "linc_qc.tsv")
      .query("space == 'fsLR'")
      .assign(sub=lambda x: x['sub'].astype(str).str.zfill(3))
      .set_index(["sub", "run"], drop=False)
)
display(linc_qc.head())

# get subs and sessions with mean_fd < 0.3. 
subs_all = linc_qc["sub"].unique()
sub_ses = {
    1: [],
    2: []
}
for run in [1, 2]:
    for sub in subs_all:
        if linc_qc.loc[(sub, run), "mean_fd"] <= 0.3: 
            sub_ses[run].append(sub)
print(f"{len(sub_ses[1])}/{len(subs_all)} subs with mean_fd <= 0.3 for run-1")
print(f"{len(sub_ses[2])}/{len(subs_all)} subs with mean_fd <= 0.3 for run-2")

# subjects with both sessions good
subs = [sub for sub in subs_all if sub in sub_ses[1] and sub in sub_ses[2]]
print(f"{len(subs)}/{len(subs_all)} subs with both sessions good")


sub  task  run space  den   mean_fd  mean_fd_post_censoring  \
sub    run                                                                   
pa1372 1    pa1372  rest    1  fsLR  91k  0.152312                0.152312   
       2    pa1372  rest    2  fsLR  91k  0.098428                0.098428   
pa1387 1    pa1387  rest    1  fsLR  91k  0.168287                0.168287   
       2    pa1387  rest    2  fsLR  91k  0.138856                0.138856   
pa1393 1    pa1393  rest    1  fsLR  91k  0.203768                0.203768   

            mean_relative_rms  max_relative_rms  mean_dvars_initial  \
sub    run                                                            
pa1372 1             0.088896          0.301342            1.245110   
       2             0.055842          0.204118            1.248755   
pa1387 1             0.100775          0.330907            1.249841   
       2             0.078597          0.567888            1.261460   
pa1393 1             0.117141          1.039210            1.268572   

            mean_dvars_final  num_dummy_volumes  num_censored_volumes  \
sub    run                                                              
pa1372 1            1.251829                  0                     0   
       2            1.253988                  0                     0   
pa1387 1            1.247582                  0                     0   
       2            1.246908                  0                     0   
pa1393 1            1.301904                  0                     0   

            num_retained_volumes  fd_dvars_correlation_initial  \
sub    run                                                       
pa1372 1                     410                      0.249939   
       2                     410                      0.193478   
pa1387 1                     410                      0.313367   
       2                     410                      0.353823   
pa1393 1                     410                      0.614906   

            fd_dvars_correlation_final  
sub    run                              
pa1372 1                      0.082654  
       2                     -0.027859  
pa1387 1                      0.026168  
       2                     -0.066287  
pa1393 1                     -0.096553

27/27 subs with mean_fd <= 0.3 for run-1
26/27 subs with mean_fd <= 0.3 for run-2
26/27 subs with both sessions good


### Connectomes

In [9]:
# only Schaefer200Subcortical
for parc in ["Schaefer200Subcortical", "Schaefer200"]:

    # load timeseries arrays for cortical and subcortical parcellations
    if parc in PARCS_CX:
        timeseries_xarray = (
            pickle.load(gzip.open(wd / "data_source" / "timeseries" / "yrsp" / f"seg-{parc}_timeseries.pkl.gz", "rb"))
            .loc[([f"sub-{sub}" for sub in subs], slice(None), slice(None)), :, :]
        )
        
    # for whole-brain parcellation, load both cortical and subcortical timeseries arrays
    else:
        timeseries_xarray = [
            (
                pickle.load(gzip.open(wd / "data_source" / "timeseries" / "yrsp" / f"seg-{p}_timeseries.pkl.gz", "rb"))
                .loc[([f"sub-{sub}" for sub in subs], slice(None), slice(None)), :, sc_parcels if p == "HCP" else slice(None)]
            )
            for p in [parc.replace("Subcortical", ""), "HCP"]
        ]
        
    # calculate connectivity
    conn, sac = connectivity_from_timeseries(
        timeseries_xarray=timeseries_xarray, 
        connectivity_measures="pearson",
        calculate_autocorrelation=True, 
        distmat=distmat[parc]
    )
    print(conn.shape)
    print("")

    # save
    with gzip.open(wd / "data_deriv" / "connectomes" / "yrsp" / f"parc-{parc}_connectivity.pkl.gz", "wb") as f:
        pickle.dump(conn, f)
    sac.to_csv(wd / "data_deriv" / "connectomes" / "yrsp" / f"parc-{parc}_sac.csv")

timeseries_xarray is a list, concatenating in across dimension 'parcel'
timeseries_xarray.shape: (52, 410, 216)
xarray_dims: ('sub-run', 'tp', 'parcel')
xarray_coords: ['sub-run', 'sub', 'run', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run']
average_by_dir: False
index_to_iterate: MultiIndex([('sub-pa1372', '01'),
            ('sub-pa1372', '02'),
            ('sub-pa1387', '01'),
            ('sub-pa1387', '02'),
            ('sub-pa1416', '01')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/52 [00:00<?, ?it/s]

pearson | ('sub-pa1372', '01') | ts: (410, 216)
pearson | ('sub-pa1372', '01') | mat: (216, 216)
pearson | ('sub-pa1372', '01') | conn: (216, 216)
(1, 52, 216, 216)

timeseries_xarray.shape: (52, 410, 200)
xarray_dims: ('sub-run', 'tp', 'parcel')
xarray_coords: ['sub-run', 'sub', 'run', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run']
average_by_dir: False
index_to_iterate: MultiIndex([('sub-pa1372', '01'),
            ('sub-pa1372', '02'),
            ('sub-pa1387', '01'),
            ('sub-pa1387', '02'),
            ('sub-pa1416', '01')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/52 [00:00<?, ?it/s]

pearson | ('sub-pa1372', '01') | ts: (410, 200)
pearson | ('sub-pa1372', '01') | mat: (200, 200)
pearson | ('sub-pa1372', '01') | conn: (200, 200)
(1, 52, 200, 200)



## DRUG: mph

### Subjects

Selection was made already

In [10]:
subs_drug_mph = sorted([
    f.name.removesuffix(".txt").split("_")[-1] 
    for f in (wd / "data_source" / "timeseries" / "drug_mph" / "schaefer200_drug").glob("*.txt")
])
print(len(subs_drug_mph), "Subjects")

30 Subjects


### Connectomes

We only have Schaefer200 data for this dataset

In [11]:
parc = "Schaefer200"
print(parc)

# We want a 3d xarray with the following structure:
# 1st dim: subjects x treatment
# 2nd dim: time points
# 3rd dim: parcels

timeseries_xarray = {}

# iterate subjects
for sub in subs_drug_mph:
    #print(sub)
    
    # iterate treatment
    for treat, treat_name in [("placebo", "placebo"), ("drug", "mph")]:
        #print(treat_name)
        
        # load timeseries
        timeseries = np.loadtxt(
            wd / "data_source" / "timeseries" / "drug_mph" / f"schaefer200_{treat}" / f"schaefer_{treat}_{sub}.txt",
            dtype=np.float32
        )
        
        # standardize
        timeseries = zscore(timeseries, axis=0)
        
        # save
        timeseries_xarray[sub, treat_name] = timeseries

# build xarray
timeseries_xarray = xr.DataArray(
    data=np.stack([d for d in timeseries_xarray.values()]),
    coords={
        "sub-treat": pd.MultiIndex.from_tuples(timeseries_xarray.keys(), names=["sub", "treat"]),
        "tp": np.arange(timeseries_xarray[subs_drug_mph[0], "placebo"].shape[0]),
        "parcel": np.loadtxt(wd / "parcellation" / f"parc-{parc}.label.csv", str, skiprows=1)
    }
)
display(timeseries_xarray)

# run connectivity estimation
conn, sac = connectivity_from_timeseries(
    timeseries_xarray=timeseries_xarray, 
    connectivity_measures="pearson",
    calculate_autocorrelation=True, 
    distmat=distmat[parc]
)

# save
with gzip.open(wd / "data_deriv" / "connectomes" / "drug_mph" / f"parc-{parc}_connectivity.pkl.gz", "wb") as f:
    pickle.dump(conn, f)
sac.to_csv(wd / "data_deriv" / "connectomes" / "drug_mph" / f"parc-{parc}_sac.csv")

Schaefer200


<xarray.DataArray (sub-treat: 60, tp: 196, parcel: 200)> Size: 9MB
array([[[-1.1002158 , -1.0615681 , -0.19844873, ...,  0.2024978 ,
         -0.5537486 ,  0.4210994 ],
        [-0.40746188,  0.1214188 ,  0.03293123, ...,  1.6898142 ,
         -0.00809112,  0.22280796],
        [-0.50723165, -0.14831942, -0.33216146, ..., -1.1710321 ,
         -1.1419667 , -1.0434592 ],
        ...,
        [-2.0658937 ,  0.15637133, -0.38888   , ..., -1.977743  ,
         -1.0127718 , -1.4369831 ],
        [ 0.45734677,  0.8044568 , -0.46408513, ..., -0.5006317 ,
         -0.34680003, -1.3097671 ],
        [ 0.7664131 ,  1.0452814 , -0.43923762, ..., -0.18842937,
          0.8212161 , -0.00728748]],

       [[ 0.3246205 , -0.4587772 , -0.36660028, ..., -0.555371  ,
         -0.4298301 ,  0.11002375],
        [ 0.3279058 ,  0.509296  ,  1.3420061 , ...,  0.0952244 ,
         -0.25687945, -0.33586198],
        [ 0.43428746,  0.25662985,  0.69916797, ...,  0.42918295,
         -1.1189069 , -1.3557532 ],
...
        [-1.755908  , -1.2301928 , -2.0958872 , ...,  0.84418833,
          0.7195958 ,  1.1537012 ],
        [-0.78859454, -0.19705568,  0.366124  , ...,  1.094964  ,
          0.94145167, -0.18966809],
        [ 1.24447   ,  1.0469487 ,  1.025838  , ..., -0.3067116 ,
          0.6217646 , -0.1756222 ]],

       [[ 1.1549134 ,  1.1134601 , -0.54871887, ...,  1.0110723 ,
          1.0268164 ,  0.5817497 ],
        [ 0.8833368 ,  3.2204108 , -0.7276134 , ..., -0.23133905,
         -0.3482659 , -0.6033321 ],
        [ 0.83107394,  2.3867488 , -0.2769301 , ...,  1.0171949 ,
          0.05322983,  0.5370664 ],
        ...,
        [-2.0872822 , -1.7693176 ,  0.55105585, ..., -1.1918294 ,
          0.73963493, -0.7672068 ],
        [-0.8145399 , -1.4903014 ,  0.5851755 , ..., -0.27052352,
          1.0725161 , -0.5818581 ],
        [-0.3116765 , -1.0259255 , -0.4026587 , ...,  0.3852511 ,
          1.1411307 ,  0.8573324 ]]], dtype=float32)
Coordinates:
  * sub-treat  (sub-treat) object 480B MultiIndex
  * sub        (sub-treat) object 480B 'DMA034' 'DMA034' ... 'DMA064' 'DMA064'
  * treat      (sub-treat) object 480B 'placebo' 'mph' ... 'placebo' 'mph'
  * tp         (tp) int64 2kB 0 1 2 3 4 5 6 7 ... 189 190 191 192 193 194 195
  * parcel     (parcel) <U43 34kB '1,hemi-L_div-Vis_lab-1' ... '200,hemi-R_di...

timeseries_xarray.shape: (60, 196, 200)
xarray_dims: ('sub-treat', 'tp', 'parcel')
xarray_coords: ['sub-treat', 'sub', 'treat', 'tp', 'parcel']
xarray_index_coords: ['sub', 'treat']
average_by_dir: False
index_to_iterate: MultiIndex([('DMA034', 'placebo'),
            ('DMA034',     'mph'),
            ('DMA035', 'placebo'),
            ('DMA035',     'mph'),
            ('DMA036', 'placebo')],
           names=['sub', 'treat'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/60 [00:00<?, ?it/s]

pearson | ('DMA034', 'placebo') | ts: (196, 200)
pearson | ('DMA034', 'placebo') | mat: (200, 200)
pearson | ('DMA034', 'placebo') | conn: (200, 200)


## DRUG: ketamine/midazolam

### Subjects

In [21]:
# QC data 
linc_qc = (
      pd.read_table(wd / "data_source" / "timeseries" / "drug_ket_mdz" / "linc_qc.tsv")
      .query("space == 'fsLR' & acq == 'infusion'")
      .assign(sub=lambda x: x['sub'].astype(str).str.zfill(3))
      .set_index(["sub", "ses"], drop=False)
)
#display(linc_qc.head())

# get subs and sessions with mean_fd <= 0.3. also keep subjects with less then all sessions
subs_drug_ket_mdz = {}
for trt in ["PLA", "KET", "MDZ"]:
    subs_drug_ket_mdz[trt] = linc_qc.query(f"mean_fd <= 0.3 & ses == '{trt}' & num_retained_volumes == 437").index
    print(f"{trt} - len(subs): {len(subs_drug_ket_mdz[trt].get_level_values('sub').unique())} of "
          f"{len(linc_qc['sub'].unique())} have mean_fd < 0.3")

PLA - len(subs): 28 of 29 have mean_fd < 0.3
KET - len(subs): 28 of 29 have mean_fd < 0.3
MDZ - len(subs): 24 of 29 have mean_fd < 0.3


### Connectomes

In [29]:
# parcellation is only Schaefer200Subcortical
parcs = ["Schaefer200", "HCP"]

# We want a 3d xarray with the following structure:
# 1st dim: subjects x treatment
# 2nd dim: time points
# 3rd dim: parcels

new_treat_names = {"KET": "ketamine", 
                   "MDZ": "midazolam", 
                   "PLA": "placebo"}

# temporary dict 
tmp = {}
tmp_windowed = {}

for parc in parcs:
    print(parc)
    tmp[parc], tmp_windowed[parc] = {}, {}
    
    for treatment in ["KET", "MDZ"]:
        print(treatment)
        
        sel = [(f"sub-{sub}", "PLA", "infusion", "fsLR") for sub, ses in subs_drug_ket_mdz["PLA"]] + \
            [(f"sub-{sub}", treatment, "infusion", "fsLR") for sub, ses in subs_drug_ket_mdz[treatment]]

        # load timeseries array
        # directly subset: subjects, placebo+treatment, fsLR space
        timeseries_xarray = (
            pickle.load(gzip.open(wd / "data_source" / "timeseries" / "drug_ket_mdz" / f"seg-{parc}_timeseries.pkl.gz", "rb"))
            .loc[sel, :, sc_parcels if parc == "HCP" else slice(None)]
            .astype(np.float32)
        )
        display(timeseries_xarray.shape, timeseries_xarray.coords)
        
        # GET WINDOWED TIMESERIES DATA
        
        # redo array and get windowed timeseries
        tmp_windowed[parc][treatment] = window_timeseries(
            xr.DataArray(
                data=timeseries_xarray.values,
                coords={
                    "sub-treat": pd.MultiIndex.from_tuples(
                        [(sub, new_treat_names[treat]) for sub, treat, _, _ in timeseries_xarray.coords["sub-ses-acq-space"].values],
                        names=["sub", "treat"]
                    ),
                    "tp": timeseries_xarray.coords["tp"].values,
                    "parcel": timeseries_xarray.coords["parcel"].values
                }
            ), 
            subindex_key="sub-treat", 
            tr=2.2
        )
        
        # we want a new xarray with the following structure:
        # 1st dim: subjects x treatment x session (session is made from continuous run, first and last 196 tps)
        # 2nd dim: time points (-> 190)
        # 3rd dim: parcels
        
        # SORT DATA in PRE/POST INFUSION SESSIONS
        
        # iterate over sessions
        timeseries_xarray_ses = {}
        for ses, tps in [("pre", slice(None, 190)), ("post", slice(-190, None))]:
            
            # redo array
            timeseries_xarray_ses[ses] = xr.DataArray(
                data=timeseries_xarray[:, tps, :].values,
                coords={
                    "sub-treat-ses": pd.MultiIndex.from_tuples(
                        [(sub, new_treat_names[treat], ses) for sub, treat, _, _ in timeseries_xarray.coords["sub-ses-acq-space"].values],
                        names=["sub", "treat", "ses"]
                    ),
                    "tp": range(190),
                    "parcel": timeseries_xarray.coords["parcel"].values
                }
            )
        
        # concat
        timeseries_xarray_ses = xr.concat([timeseries_xarray_ses["pre"], timeseries_xarray_ses["post"]], dim="sub-treat-ses")
        display(timeseries_xarray_ses.shape, timeseries_xarray_ses.coords)
            
        # save in temp dict
        tmp[parc][treatment] = timeseries_xarray_ses

        
    
# RUN
for treatment in ["KET", "MDZ"]:
    print(treatment)
    
    # PRE/POST INFUSION SESSIONS
    print("pre/post infusion sessions")
    
    # load timeseries array
    timeseries_xarray = [tmp[parc][treatment] for parc in parcs]
    if len(timeseries_xarray) == 1:
        timeseries_xarray = timeseries_xarray[0]
        
    # get parc name
    p = "".join([parc.replace("HCP", "Subcortical") for parc in parcs])
    
    # run connectivity estimation
    conn, sac = connectivity_from_timeseries(
        timeseries_xarray=timeseries_xarray, 
        connectivity_measures="pearson",
        calculate_autocorrelation=True, 
        distmat=distmat[p]
    )

    # # save
    with gzip.open(wd / "data_deriv" / "connectomes" / f"drug_{treatment.lower()}" / f"parc-{p}_connectivity.pkl.gz", "wb") as f:
        pickle.dump(conn, f)
    sac.to_csv(wd / "data_deriv" / "connectomes" / f"drug_{treatment.lower()}" / f"parc-{p}_sac.csv")
    
    # add sac for Schaefer200
    sac = calculate_sac(conn[:,:,:-16, :-16], distmat["Schaefer200"])
    sac.to_csv(wd / "data_deriv" / "connectomes" / f"drug_{treatment.lower()}" / f"parc-Schaefer200_sac.csv")
        
    # WINDOWED TIMESERIES
    print("windowed timeseries")
    
    # load timeseries array
    timeseries_xarray = [tmp_windowed[parc][treatment] for parc in parcs]
    if len(timeseries_xarray) == 1:
        timeseries_xarray = timeseries_xarray[0]
        
    # run connectivity estimation
    conn = connectivity_from_timeseries(timeseries_xarray=timeseries_xarray, connectivity_measures=["pearson"])

    # # save
    with gzip.open(wd / "data_deriv" / "connectomes" / f"drug_{treatment.lower()}" / f"parc-{p}_windowconnectivity.pkl.gz", "wb") as f:
        pickle.dump(conn, f)
    

Schaefer200
KET


(56, 437, 200)

Coordinates:
  * sub-ses-acq-space  (sub-ses-acq-space) object 448B MultiIndex
  * sub                (sub-ses-acq-space) object 448B 'sub-001' ... 'sub-038'
  * ses                (sub-ses-acq-space) object 448B 'PLA' 'PLA' ... 'KET'
  * acq                (sub-ses-acq-space) object 448B 'infusion' ... 'infusion'
  * space              (sub-ses-acq-space) object 448B 'fsLR' 'fsLR' ... 'fsLR'
  * tp                 (tp) int64 3kB 0 1 2 3 4 5 6 ... 431 432 433 434 435 436
  * parcel             (parcel) object 2kB 'LH_Vis_1' ... 'RH_Default_pCunPCC_3'

60 s windows with 10 s steps @ tr = 2.2s: 28 elements per window and 5 element steps


(112, 190, 200)

...

pearson | ('sub-001_win-0', 'placebo') | ts: (29, 216)
pearson | ('sub-001_win-0', 'placebo') | mat: (216, 216)
pearson | ('sub-001_win-0', 'placebo') | conn: (216, 216)
MDZ
pre/post infusion sessions
timeseries_xarray is a list, concatenating in across dimension 'parcel'
timeseries_xarray.shape: (104, 190, 216)
xarray_dims: ('sub-treat-ses', 'tp', 'parcel')
xarray_coords: ['sub-treat-ses', 'sub', 'treat', 'ses', 'tp', 'parcel']
xarray_index_coords: ['sub', 'treat', 'ses']
average_by_dir: False
index_to_iterate: MultiIndex([('sub-001', 'placebo', 'pre'),
            ('sub-002', 'placebo', 'pre'),
            ('sub-003', 'placebo', 'pre'),
            ('sub-004', 'placebo', 'pre'),
            ('sub-006', 'placebo', 'pre')],
           names=['sub', 'treat', 'ses'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/104 [00:00<?, ?it/s]

pearson | ('sub-001', 'placebo', 'pre') | ts: (190, 216)
pearson | ('sub-001', 'placebo', 'pre') | mat: (216, 216)
pearson | ('sub-001', 'placebo', 'pre') | conn: (216, 216)
windowed timeseries
timeseries_xarray is a list, concatenating in across dimension 'parcel'
timeseries_xarray.shape: (4264, 29, 216)
xarray_dims: ('sub-treat', 'tp', 'parcel')
xarray_coords: ['sub-treat', 'sub', 'treat', 'tp', 'parcel']
xarray_index_coords: ['sub', 'treat']
average_by_dir: False
index_to_iterate: MultiIndex([('sub-001_win-0', 'placebo'),
            ('sub-002_win-0', 'placebo'),
            ('sub-003_win-0', 'placebo'),
            ('sub-004_win-0', 'placebo'),
            ('sub-006_win-0', 'placebo')],
           names=['sub', 'treat'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/4264 [00:00<?, ?it/s]

pearson | ('sub-001_win-0', 'placebo') | ts: (29, 216)
pearson | ('sub-001_win-0', 'placebo') | mat: (216, 216)
pearson | ('sub-001_win-0', 'placebo') | conn: (216, 216)


## DRUG: risperidone

### Subjects

In [4]:
# QC data 
linc_qc = (
      pd.read_table(wd / "data_source" / "timeseries" / "drug_risp" / "linc_qc.tsv")
      .query("space == 'fsLR'")
      .astype({"sub": str})
      .set_index(["sub", "ses"], drop=False)
      .merge(
            pd.read_table(wd / "data_source" / "timeseries" / "drug_risp" / "participants.tsv") \
                  .assign(ses=lambda x: x["session_id"].str.replace("ses-", ""),
                          sub=lambda x: x["participant_id"].str.replace("sub-", "")) \
                  .set_index(["sub", "ses"], drop=True), 
            left_index=True, 
            right_index=True
      )
)

# drop sub-1003 because errors in processing
linc_qc = linc_qc.loc[linc_qc["sub"] != "1003"]

# display
#display(linc_qc.head())

# get subs and sessions with mean_fd <= 0.3. also keep subjects with less then all sessions
drug_risp_subs_ses = {}
for trt in ["placebo", "risp_low", "risp_high"]:
    drug_risp_subs_ses[trt] = linc_qc.query(f"mean_fd <= 0.3 & treatment == '{trt}'").index
    print(f"len(subs): {len(drug_risp_subs_ses[trt].get_level_values('sub').unique())} of "
          f"{len(linc_qc['sub'].unique())} have >= 1 session with mean_fd < 0.3")

len(subs): 17 of 20 have >= 1 session with mean_fd < 0.3
len(subs): 19 of 20 have >= 1 session with mean_fd < 0.3
len(subs): 17 of 20 have >= 1 session with mean_fd < 0.3


### Connectomes

In [5]:
# only Schaefer200Subcortical
parcs = ["Schaefer200", "HCP"]

sel = [(f"sub-{sub}", ses, "fsLR") for sub, ses in np.concatenate(list(drug_risp_subs_ses.values()))]
# We want a 3d xarray with the following structure:
# 1st dim: subjects x treatment
# 2nd dim: time points
# 3rd dim: parcels

tmp = []
for p in parcs:
    timeseries_xarray = (
        pickle.load(gzip.open(wd / "data_source" / "timeseries" / "drug_risp" / f"seg-{p}_timeseries.pkl.gz", "rb"))
        .loc[sel, :, sc_parcels if p == "HCP" else slice(None)]
    )
    timeseries_xarray = xr.DataArray(
        data=timeseries_xarray.values,
        coords={
            "sub-treat": pd.MultiIndex.from_tuples(
                [(sub, linc_qc.loc[(sub.replace("sub-", ""), ses), "treatment"]) for sub, ses, _ in sel],
                names=["sub", "treat"]
            ),
            "tp": timeseries_xarray.coords["tp"].values,
            "parcel": timeseries_xarray.coords["parcel"].values
        },
    )
    tmp.append(timeseries_xarray)

# calculate connectivity
conn, sac = connectivity_from_timeseries(
    timeseries_xarray=tmp, 
    connectivity_measures="pearson",
    calculate_autocorrelation=True, 
    distmat=distmat["Schaefer200Subcortical"]
)
print(conn.shape)
print("")

# save
with gzip.open(wd / "data_deriv" / "connectomes" / "drug_risp" / f"parc-Schaefer200Subcortical_connectivity.pkl.gz", "wb") as f:
    pickle.dump(conn, f)
sac.to_csv(wd / "data_deriv" / "connectomes" / "drug_risp" / f"parc-Schaefer200Subcortical_sac.csv")

# add sac for Schaefer200
sac = calculate_sac(conn[:,:,:-16, :-16], distmat["Schaefer200"])
sac.to_csv(wd / "data_deriv" / "connectomes" / "drug_risp" / f"parc-Schaefer200_sac.csv")

timeseries_xarray is a list, concatenating in across dimension 'parcel'
timeseries_xarray.shape: (53, 300, 216)
xarray_dims: ('sub-treat', 'tp', 'parcel')
xarray_coords: ['sub-treat', 'sub', 'treat', 'tp', 'parcel']
xarray_index_coords: ['sub', 'treat']
average_by_dir: False
index_to_iterate: MultiIndex([('sub-1004', 'placebo'),
            ('sub-1005', 'placebo'),
            ('sub-1006', 'placebo'),
            ('sub-1007', 'placebo'),
            ('sub-1008', 'placebo')],
           names=['sub', 'treat'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/53 [00:00<?, ?it/s]

pearson | ('sub-1004', 'placebo') | ts: (300, 216)
pearson | ('sub-1004', 'placebo') | mat: (216, 216)
pearson | ('sub-1004', 'placebo') | conn: (216, 216)
(1, 53, 216, 216)



## HCP-EP

### Subjects

In [5]:
# pheno data
pheno_ep = pd.read_csv(wd / "data_deriv" / "pheno" / "hcp_ep" / "hcpep_pheno.csv", index_col=0)

# QC data 
qc_ep = pd.read_csv(wd / "data_deriv" / "pheno" / "hcp_ep" / "hcpep_qc.csv", index_col=0)

# subjects that are in pheno and qc
subs_ep_avail = list(set(pheno_ep.index).intersection(set(qc_ep.index)))
print(f"{len(subs_ep_avail)}/{len(pheno_ep)} subs in pheno and qc")

# get subs with mean_fd <= 0.3 for both directions in both runs
subs_ep = qc_ep.loc[subs_ep_avail].query("qc_pass==True").index.tolist()
print(f"{len(subs_ep)}/{len(subs_ep_avail)} subs with mean_fd <= 0.3 for both directions")

# check
pheno_ep.loc[subs_ep, "dx"].value_counts()

159/251 subs in pheno and qc
151/159 subs with mean_fd <= 0.3 for both directions


dx
SCZ     96
CTRL    55
Name: count, dtype: int64

### Connectomes

In [6]:
for parc in ["Schaefer200", "Schaefer200Subcortical"]:
    print(parc)
    

    # load timeseries arrays for cortical and subcortical parcellations
    if parc in PARCS_CX:
        timeseries_xarray = (
            pickle.load(gzip.open(wd / "data_source" / "timeseries" / "hcp_ep" / f"seg-{parc}_timeseries.pkl.gz", "rb"))
            .loc[(subs_ep, slice(None), slice(None)), :, :]
        )
        
    # for whole-brain parcellation, load both cortical and subcortical timeseries arrays
    else:
        timeseries_xarray = [
            (
                pickle.load(gzip.open(wd / "data_source" / "timeseries" / "hcp_ep" / f"seg-{p}_timeseries.pkl.gz", "rb"))
                .loc[(subs_ep, slice(None), slice(None)), :, sc_parcels if p == "HCP" else slice(None)]
            )
            for p in [parc.replace("Subcortical", ""), "HCP"]
        ]
        
    # calculate connectivity
    conn, sac = connectivity_from_timeseries(
        timeseries_xarray=timeseries_xarray, 
        connectivity_measures="pearson",
        calculate_autocorrelation=True, 
        distmat=distmat[parc]
    )
    print(conn.shape)
    
    # adjust index to sub-dx
    conn = xr.DataArray(
        conn.values,
        coords={
            "measure": ["pearson"],
            "sub-dx": pd.MultiIndex.from_tuples(
                [(sub, pheno_ep.loc[sub, "dx"]) for sub in conn.coords["sub"].values],
                names=["sub", "dx"]
            ),
            "parcelx": conn.coords["parcelx"].values,
            "parcely": conn.coords["parcely"].values,
        }
    )
    sac = (
        sac.reset_index(["measure", "sub"])
        .assign(dx=lambda x: pheno_ep.loc[x["sub"], "dx"].values)
        .set_index(["measure", "sub", "dx"])
    )
    
    # save
    with gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ep" / f"parc-{parc}_connectivity.pkl.gz", "wb") as f:
        pickle.dump(conn, f)
    sac.to_csv(wd / "data_deriv" / "connectomes" / "hcp_ep" / f"parc-{parc}_sac.csv")

Schaefer200
timeseries_xarray.shape: (302, 410, 200)
xarray_dims: ('sub-run-dir', 'tp', 'parcel')
xarray_coords: ['sub-run-dir', 'sub', 'run', 'dir', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run', 'dir']
average_by_dir: True
index_to_iterate: MultiIndex([('sub-3027', '1'),
            ('sub-1010', '1'),
            ('sub-3025', '1'),
            ('sub-1040', '1'),
            ('sub-1021', '1')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/151 [00:00<?, ?it/s]

pearson | ('sub-3027', '1') | ts (RL, LR): (410, 200) (410, 200)
pearson | ('sub-3027', '1') | mat: (200, 200)
pearson | ('sub-3027', '1') | mat: (200, 200)
pearson | ('sub-3027', '1') | conn: (200, 200)
(1, 151, 200, 200)
Schaefer200Subcortical
timeseries_xarray is a list, concatenating in across dimension 'parcel'
timeseries_xarray.shape: (302, 410, 216)
xarray_dims: ('sub-run-dir', 'tp', 'parcel')
xarray_coords: ['sub-run-dir', 'sub', 'run', 'dir', 'tp', 'parcel']
xarray_index_coords: ['sub', 'run', 'dir']
average_by_dir: True
index_to_iterate: MultiIndex([('sub-3027', '1'),
            ('sub-1010', '1'),
            ('sub-3025', '1'),
            ('sub-1040', '1'),
            ('sub-1021', '1')],
           names=['sub', 'run'])
Connectivity measure: pearson with parameter: None


  0%|          | 0/151 [00:00<?, ?it/s]

pearson | ('sub-3027', '1') | ts (RL, LR): (410, 216) (410, 216)
pearson | ('sub-3027', '1') | mat: (216, 216)
pearson | ('sub-3027', '1') | mat: (216, 216)
pearson | ('sub-3027', '1') | conn: (216, 216)
(1, 151, 216, 216)
